# modeling_v6 — 시퀀스 모델 (1D-CNN, step 시퀀스 직접 학습)

**목표**: tabular 프레임(v1~v5)이 수렴한 RMSE ~60 천장 돌파 → 목표 ~40.

**왜 시퀀스인가**: v5에서 step×센서 상호작용이 실재함을 확인(C17 상관 step별 0.19~0.80)했으나, 집계/row-level 트리는 여전히 WF 평균에 의존해 천장을 못 넘음. step **순서 구조**를 직접 학습하는 딥러닝이 유일한 돌파 후보.

**설계**
- WF당 step 시퀀스를 시간순(C40, 원본순서와 99% 일치)으로 정렬 → 고정 길이 `MAX_LEN=16` (p99=13, 초과분 truncate), zero-padding + mask.
- 입력 채널 = 36 센서(표준화) + C7 one-hot 5 = **41 채널**.
- C23(28종 Recipe)은 WF-level 임베딩으로 pooled feature에 concat (v5에서 유효 확인).
- 타깃 C65 표준화 후 학습, 예측 시 역변환.
- **GroupKFold(C64)** 5-Fold, WF-level RMSE 평가.

**비교 기준**: v5 Valid 61.38 / Test 60.52, v3 Test 60.51.

## Cell 1 — imports & 설정

In [1]:
import numpy as np, pandas as pd, json, time, warnings
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings("ignore")

SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

DATA_DIR=Path("../문제1(하)"); ANS_DIR=Path("../문제1_하_answer")
OUT=Path("outputs"); OUT.mkdir(exist_ok=True)
WF_ID,TARGET,N_FOLDS = "C64","C65",5
MAX_LEN = 16

SENSORS=['C1','C3','C4','C5','C8','C9','C11','C12','C15','C16','C17','C18','C19','C25',
 'C27','C31','C32','C33','C42','C44','C45','C46','C48','C49','C50','C51','C52','C54',
 'C56','C57','C58','C59','C60','C61','C62','C63']
C7_LEVELS=[1.0,4.0,5.0,6.0,7.0]   # step 종류
print("센서 채널:", len(SENSORS), "| +C7 one-hot:", len(C7_LEVELS), "= 총", len(SENSORS)+len(C7_LEVELS))

device: cpu
센서 채널: 36 | +C7 one-hot: 5 = 총 41


## Cell 2 — 데이터 로드

In [2]:
train=pd.read_csv(DATA_DIR/"train_data.csv")
valid_X=pd.read_csv(DATA_DIR/"valid_X.csv"); test_X=pd.read_csv(DATA_DIR/"test_X.csv")
valid_Yp=pd.read_csv(DATA_DIR/"valid_Y_problem.csv"); test_Yp=pd.read_csv(DATA_DIR/"test_Y_problem.csv")
valid_Ya=pd.read_csv(ANS_DIR/"valid_Y_answer.csv").set_index("C64")["C65"]
test_Ya =pd.read_csv(ANS_DIR/"test_Y_answer.csv").set_index("C64")["C65"]
cols=[c for c in train.columns if c!=TARGET]
valid_X=valid_X[cols]; test_X=test_X[cols]
print(train.shape, valid_X.shape, test_X.shape)

(123614, 65) (20577, 64) (20510, 64)


## Cell 3 — 표준화 통계 & C23 인덱스 매핑 (train 기준)

In [3]:
# 센서 표준화 통계 (train)
mu = train[SENSORS].mean(); sd = train[SENSORS].std().replace(0,1.0)
# 타깃 표준화
y_mu, y_sd = train[TARGET].mean(), train[TARGET].std()
# C23 인덱스 (train에 없는 값은 0=UNK; 재탐색 결과 valid/test 미관측 0개)
c23_cats = sorted(train["C23"].unique())
c23_map = {v:i+1 for i,v in enumerate(c23_cats)}   # 0=UNK
N_C23 = len(c23_map)+1
print("C23 카테고리:", len(c23_cats), "| y_mu %.1f y_sd %.1f"%(y_mu,y_sd))

C23 카테고리: 28 | y_mu 972.0 y_sd 261.8


## Cell 4 — WF → 3D 텐서 변환기

In [4]:
def to_tensors(df, has_target=True):
    df = df.copy()
    df["_dt"] = pd.to_datetime(df["C40"])
    df = df.sort_values([WF_ID,"_dt"]).reset_index(drop=True)
    # 센서 표준화
    S = ((df[SENSORS]-mu)/sd).values.astype(np.float32)
    # C7 one-hot
    c7 = df["C7"].values
    OH = np.zeros((len(df), len(C7_LEVELS)), np.float32)
    for j,lv in enumerate(C7_LEVELS): OH[c7==lv, j]=1.0
    feats = np.concatenate([S, OH], axis=1)          # (rows, 41)
    C = feats.shape[1]

    wfs = df[WF_ID].values
    # 각 WF의 시작 offset과 길이 (df가 WF로 정렬돼 있으므로 연속 블록)
    uniq, first_idx, counts = np.unique(wfs, return_index=True, return_counts=True)
    order = np.argsort(first_idx)            # 등장 순서 보존
    uniq, first_idx, counts = uniq[order], first_idx[order], counts[order]

    N=len(uniq)
    X = np.zeros((N, MAX_LEN, C), np.float32)
    M = np.zeros((N, MAX_LEN), np.float32)   # mask (1=유효)
    for i in range(N):
        st=first_idx[i]; k=min(counts[i], MAX_LEN)
        X[i,:k]=feats[st:st+k]; M[i,:k]=1.0  # 앞쪽 k step 사용 (시간 이른 순)
    c23_ser = pd.Series(df["C23"].values[first_idx], index=uniq)
    c23 = c23_ser.map(c23_map).fillna(0).astype(int).values
    out = {"X":X,"M":M,"C23":c23,"wf":uniq}
    if has_target:
        out["y"] = df[TARGET].values[first_idx].astype(np.float32)
    return out

t0=time.time()
TR=to_tensors(train,True); VA=to_tensors(valid_X,False); TE=to_tensors(test_X,False)
print("변환 완료 %.1fs | X:"%(time.time()-t0), TR["X"].shape)

변환 완료 0.6s | X: (11939, 16, 41)


## Cell 5 — 1D-CNN 모델 정의

In [5]:
class CNN1D(nn.Module):
    def __init__(self, in_ch, n_c23, emb=8, ch=64, p=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_c23, emb)
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch, ch, 3, padding=1),    nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch, ch*2, 3, padding=1),  nn.BatchNorm1d(ch*2), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(ch*2 + emb, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 1))
    def forward(self, x, m, c23):
        # x:(B,L,C) -> (B,C,L)
        h = self.conv(x.transpose(1,2))               # (B, ch2, L)
        m2 = m.unsqueeze(1)                            # (B,1,L)
        # masked mean+max pooling
        s = (h*m2).sum(-1) / m2.sum(-1).clamp(min=1)   # (B,ch2)
        hm = h.masked_fill(m2==0, -1e9).max(-1).values # (B,ch2)
        pooled = s  # mean pooling 사용 (max는 padding 민감 → 필요시 concat)
        z = torch.cat([pooled, self.emb(c23)], dim=1)
        return self.head(z).squeeze(-1)

## Cell 6 — 학습 루프 (GroupKFold 5-Fold)

In [6]:
def make_loader(d, idx, bs, shuffle, with_y=True):
    X=torch.tensor(d["X"][idx]); M=torch.tensor(d["M"][idx]); C=torch.tensor(d["C23"][idx])
    if with_y:
        y=torch.tensor(((d["y"][idx]-y_mu)/y_sd).astype(np.float32))
        ds=TensorDataset(X,M,C,y)
    else:
        ds=TensorDataset(X,M,C)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

def predict(model, d, idx=None):
    if idx is None: idx=np.arange(len(d["wf"]))
    model.eval(); preds=[]
    with torch.no_grad():
        for X,M,C in make_loader(d, idx, 512, False, with_y=False):
            X,M,C=X.to(DEVICE),M.to(DEVICE),C.to(DEVICE)
            preds.append(model(X,M,C).cpu().numpy())
    return np.concatenate(preds)*y_sd + y_mu

EPOCHS, BS, PATIENCE = 60, 256, 8
gkf=GroupKFold(N_FOLDS)
groups=TR["wf"]
oof=np.zeros(len(TR["wf"]))
va_pred=np.zeros(len(VA["wf"])); te_pred=np.zeros(len(TE["wf"]))

for k,(tri,vai) in enumerate(gkf.split(TR["wf"], TR["y"], groups)):
    model=CNN1D(len(SENSORS)+len(C7_LEVELS), N_C23).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    lossf=nn.MSELoss()
    tl=make_loader(TR,tri,BS,True); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for X,M,C,y in tl:
            X,M,C,y=[t.to(DEVICE) for t in (X,M,C,y)]
            opt.zero_grad(); out=model(X,M,C); loss=lossf(out,y); loss.backward(); opt.step()
        # fold val RMSE (원 스케일)
        vp=predict(model,TR,vai); rmse=np.sqrt(mean_squared_error(TR["y"][vai], vp))
        sch.step(rmse)
        if rmse<best-1e-3: best=rmse; best_state={kk:v.cpu().clone() for kk,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state)
    oof[vai]=predict(model,TR,vai)
    va_pred+=predict(model,VA)/N_FOLDS
    te_pred+=predict(model,TE)/N_FOLDS
    print(f"fold{k+1}: best_val_rmse={best:.4f} (ep~{ep+1})")

fold1: best_val_rmse=69.8748 (ep~29)
fold2: best_val_rmse=68.3413 (ep~25)
fold3: best_val_rmse=68.5220 (ep~29)
fold4: best_val_rmse=67.8283 (ep~35)
fold5: best_val_rmse=70.0147 (ep~29)


## Cell 7 — 평가 & 제출 저장

In [7]:
oof_rmse=np.sqrt(mean_squared_error(TR["y"], oof))
va_s=pd.Series(va_pred, index=VA["wf"]); te_s=pd.Series(te_pred, index=TE["wf"])
valid_rmse=np.sqrt(mean_squared_error(valid_Ya.loc[va_s.index], va_s))
test_rmse =np.sqrt(mean_squared_error(test_Ya.loc[te_s.index],  te_s))
print(f"CV OOF RMSE : {oof_rmse:.4f}")
print(f"Valid  RMSE : {valid_rmse:.4f}   (v5 61.38)")
print(f"Test   RMSE : {test_rmse:.4f}   (v5 60.52 / v3 60.51)")

vs=valid_Yp.copy(); vs["C65"]=vs["C64"].map(va_s); vs.to_csv(OUT/"valid_Y_submit.csv",index=False)
ts=test_Yp.copy();  ts["C65"]=ts["C64"].map(te_s); ts.to_csv(OUT/"test_Y_submit.csv",index=False)
json.dump({"oof":round(float(oof_rmse),4),"valid":round(float(valid_rmse),4),"test":round(float(test_rmse),4)},
          open(OUT/"results.json","w"), indent=2, ensure_ascii=False)
print("저장 완료")

CV OOF RMSE : 68.9216
Valid  RMSE : 68.2649   (v5 61.38)
Test   RMSE : 66.0352   (v5 60.52 / v3 60.51)
저장 완료


## Cell 8 (선택) — LSTM 변형으로 확장

1D-CNN 결과 확인 후, `CNN1D`를 아래 LSTM으로 교체해 Cell 6~7 재실행하면 시퀀스 순서 의존성을 더 직접적으로 학습할 수 있습니다.

```python
class LSTMReg(nn.Module):
    def __init__(self, in_ch, n_c23, emb=8, hid=64, p=0.2):
        super().__init__()
        self.emb=nn.Embedding(n_c23, emb)
        self.lstm=nn.LSTM(in_ch, hid, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=p)
        self.head=nn.Sequential(nn.Linear(hid*2+emb,128), nn.ReLU(),
                                nn.Dropout(p), nn.Linear(128,1))
    def forward(self,x,m,c23):
        lengths=m.sum(1).clamp(min=1).long().cpu()
        packed=nn.utils.rnn.pack_padded_sequence(x,lengths,batch_first=True,enforce_sorted=False)
        out,_=self.lstm(packed)
        out,_=nn.utils.rnn.pad_packed_sequence(out,batch_first=True)
        m2=m.unsqueeze(-1)
        pooled=(out*m2).sum(1)/m2.sum(1).clamp(min=1)
        return self.head(torch.cat([pooled,self.emb(c23)],1)).squeeze(-1)
```
Cell 6에서 `model=CNN1D(...)` → `model=LSTMReg(len(SENSORS)+len(C7_LEVELS), N_C23).to(DEVICE)` 로 교체.